# 02 - H-Optimus-1 Image Embedding From Full Cellpose Xenium Cell Boundaries, RNA-Aligned BLEEP Image Clustering, and Morphology Review

This notebook uses the full Cellpose Xenium all-cell source for `AP2320a` and `AP0921a`, configured in `config/cellpose_samples.tsv`. The expected input is a Cellpose export laid out like a Xenium output directory: `cells.csv.gz`, `cell_boundaries.csv.gz`, and `cell_feature_matrix/`. The authoritative cell IDs, H&E patch centers, and dotted patch outlines come from `cell_boundaries.csv.gz`. RNA counts come from the matching Xenium-style `cell_feature_matrix`. Registered H&E images are Palom outputs listed in the same config table.

For another dataset, this is the section to adapt: substitute the Cellpose boundary file with your Baysor cell-boundary export, and substitute the Xenium `cell_feature_matrix` with the matching spatial gene-expression object for the same cells. The important requirement is not the segmentation software itself; it is that the boundary table, RNA count matrix, and registered H&E image share the same cell IDs and coordinate system, or can be converted into that shared convention before patch extraction.

If you are using Baysor or another segmentation source, do not filter cells by a minimum transcript count in this notebook. Neutrophils can have very low transcript counts, so transcript-count filtering at the morphology-review stage can remove the cells we are trying to recover. Notebook 3 performs the transcript-count filtering later: morphology-confirmed neutrophils are rescued first, then non-neutrophils are filtered.

The workflow keeps all cells that have drawable H&E patches, reuses existing H-Optimus-1 H&E embeddings when present, aligns raw image embeddings to Xenium-style RNA PCA10 with the BLEEP-style soft-target contrastive alignment, and clusters only the RNA-aligned BLEEP image embedding with Harmony by sample for manual neutrophil review. H&E QC filtering and Muon WNN are not run in this notebook.

## Tested system and installation

This tutorial was curated on host `528nc64-l` running Ubuntu 24.04.3 LTS, Linux kernel `6.14.0-36-generic`, with 32 logical CPU cores and 125 GiB RAM. The project kernel used for Notebooks 2 and 3 is `.venv/bin/python` with Python 3.13.9.

Core package versions in the working tutorial environment include `numpy 2.4.4`, `pandas 2.3.3`, `scanpy 1.12.1`, `anndata 0.12.14`, `muon 0.1.7`, `harmonypy 2.0.0`, `torch 2.11.0+cu130`, `tifffile 2026.5.2`, `scikit-learn 1.8.0`, and `umap-learn 0.5.12`.

Create the tutorial Python kernel from the project root:

```bash
cd HEnium_tutorial_2sample
python -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r envs/requirements-hennium-python.txt
python -m pip install ipykernel
python -m ipykernel install --user --name henium-tutorial --display-name "HEnium tutorial (.venv)"
```

Notebook 2 uses the gated `bioptimus/H-optimus-1` model. For fresh H-Optimus inference, request access on Hugging Face and either run `huggingface-cli login` or set `HUGGINGFACE_HUB_TOKEN`. If `he_embeddings.npy` already exists, the notebook reuses it and does not require a fresh model download.


Input configuration for this notebook:

- `config/cellpose_samples.tsv`: sample ID, Xenium-format Cellpose output directory, and registered H&E path.
- `config/hoptimus_cellpose_allcells.yaml`: patch size, H-Optimus model ID, RNA PCA settings, contrastive alignment settings, and target-40 image-clustering settings.
- `config/tutorial_paths.yaml`: project paths and shared parameters.

Reasoning for the input format: using a Xenium-like directory gives the notebook a stable contract: cell metadata in `cells.csv.gz`, boundaries in `cell_boundaries.csv.gz`, and counts in `cell_feature_matrix/`. A Baysor or other segmentation result can be used if it is converted to the same three pieces with matching cell IDs.

In [ ]:
import importlib.util
import os
import sys

required_python = ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'tifffile', 'yaml', 'scanpy', 'anndata', 'sklearn', 'torch', 'timm']
missing = [pkg for pkg in required_python if importlib.util.find_spec(pkg) is None]
print('Python executable:', sys.executable)
print('Missing Python packages:', missing if missing else 'none')
print('HF token set:', bool(os.environ.get('HUGGINGFACE_HUB_TOKEN') or os.environ.get('HF_TOKEN')))

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, IFrame, display

def find_project_dir() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'config/tutorial_paths.yaml').exists():
            return candidate
    raise RuntimeError('Could not find project root containing config/tutorial_paths.yaml')

def project_path(value) -> Path:
    p = Path(value)
    return p if p.is_absolute() else PROJECT / p

def resolve_common_paths(cfg: dict) -> dict:
    for key in ['registration_sample_table', 'cellpose_sample_table', 'hoptimus_cellpose_config', 'input_dir', 'seurat_rds', 'subtype_reference_dir']:
        if key in cfg and cfg[key] is not None:
            cfg[key] = str(project_path(cfg[key]))
    if 'results' in cfg:
        cfg['results'] = {k: str(project_path(v)) for k, v in cfg['results'].items()}
    return cfg

PROJECT = find_project_dir()
CONFIG = PROJECT / 'config/tutorial_paths.yaml'
cfg = resolve_common_paths(yaml.safe_load(CONFIG.read_text()))
hoptimus_cfg = yaml.safe_load(Path(cfg['hoptimus_cellpose_config']).read_text())
PYTHON = PROJECT / '.venv/bin/python'
python = str(PYTHON) if PYTHON.exists() else 'python'
image_results = Path(cfg['results']['image'])
tables_dir = image_results / 'tables'
figures_dir = image_results / 'figures'
patch_dir = image_results / 'he_patches'
source_name = hoptimus_cfg['source']['name']
source_dir = image_results / 'embeddings' / source_name
filtered_source = source_dir  # compatibility alias: all downstream outputs use the full all-cell source.
cellpose_boundaries_dir = image_results / 'tables/cellpose_boundaries_allcells'
boundaries_dir = cellpose_boundaries_dir
rna_aligned_cluster_dir = image_results / 'image_rna_aligned_bleep_harmony_target40_allcells'
for d in [tables_dir, figures_dir, patch_dir, source_dir, cellpose_boundaries_dir, rna_aligned_cluster_dir]:
    d.mkdir(parents=True, exist_ok=True)
print('Image results:', image_results)
print('Full all-cell source:', source_dir)
print('Cellpose boundary output:', cellpose_boundaries_dir)
print('Python for scripts:', python)

## 1. Build or reuse the full Cellpose-boundary H-Optimus source

The prepared source is built from the Cellpose Xenium output folders listed in `config/cellpose_samples.tsv`. Cellpose `cell_boundaries.csv.gz` is the only geometry source: its polygon centroids define H&E patch centers, and the same polygon vertices are later drawn as dotted boundaries on patch review PDFs. RNA counts are loaded from each sample's `cell_feature_matrix`, so low-transcript cells are retained at this stage rather than filtered away before morphology review.

In [ ]:
cellpose_table = pd.read_csv(cfg['cellpose_sample_table'], sep='	')
analysis_samples = cfg['parameters'].get('analysis_samples', cellpose_table['sample_id'].astype(str).tolist())
cellpose_table = cellpose_table[cellpose_table['sample_id'].astype(str).isin(analysis_samples)].reset_index(drop=True)
cellpose_samples = []
for row in cellpose_table.to_dict('records'):
    item = {
        'sample_id': str(row['sample_id']),
        'xenium_dir': str(project_path(row['xenium_dir'])),
        'he_image': str(project_path(row['registered_he_ome_tif'])),
    }
    for key in ['xenium_dir', 'he_image']:
        if not Path(item[key]).exists():
            raise FileNotFoundError(f"{item['sample_id']} missing {key}: {item[key]}")
    cellpose_samples.append(item)

patch_cfg = hoptimus_cfg['patch']
image_embedding_cfg = hoptimus_cfg['image_embedding']
alignment_cfg = hoptimus_cfg['alignment']
clustering_cfg = hoptimus_cfg['image_clustering']
source_cfg = hoptimus_cfg['source']

hcfg = {
    'project_name': 'henium_tutorial_cellpose_ap2320a_ap0921a_hoptimus_bleep_image_only',
    'samples': [],
    'patch': {
        'patch_size_px': int(patch_cfg['patch_size_px']),
        'upsample_factor': float(patch_cfg['upsample_factor']),
    },
    'image_embedding': dict(image_embedding_cfg),
    'alignment': {
        'latent_dim': int(alignment_cfg['latent_dim']),
        'batch_size': int(alignment_cfg['batch_size']),
        'max_epochs': int(alignment_cfg['max_epochs']),
        'early_stop_patience': int(alignment_cfg['early_stop_patience']),
    },
    'integration': {
        'run_harmony': True,
        'harmony_key': clustering_cfg['harmony_key'],
        'embedding_file': 'aligned_image.npy',
        'umap_min_dist': float(clustering_cfg['umap_min_dist']),
        'umap_n_neighbors': int(clustering_cfg['neighbors']),
        'random_state': int(cfg['parameters']['random_seed']),
    },
}
for s in cellpose_samples:
    hcfg['samples'].append({
        'sample_id': s['sample_id'],
        'xenium_dir': s['xenium_dir'],
        'he_image': s['he_image'],
        'he_um_per_px_override': float(patch_cfg['he_um_per_px_override']),
    })
run_cfg = image_results / 'hoptimus_cellpose_harmony_config.yaml'
run_cfg.write_text(yaml.safe_dump(hcfg, sort_keys=False))
print('Configured Cellpose all-cell samples:')
display(cellpose_table)
print(run_cfg.read_text())

In [ ]:
prepare_summary = source_dir / 'prepare_summary.json'
he_summary = source_dir / 'he_embedding_summary.json'
quick_max_cells_per_sample = int(source_cfg.get('max_cells_per_sample', 0))

if not prepare_summary.exists():
    sample_args = []
    for s in cellpose_samples:
        sample_args.extend(['--sample', f"{s['sample_id']}|{s['xenium_dir']}|{s['he_image']}"])
    cmd = [
        python, str(PROJECT / 'scripts/rna_wnn/prepare_cellpose_xenium_source.py'),
        *sample_args,
        '--outdir', str(source_dir),
        '--boundaries-dir', str(cellpose_boundaries_dir),
        '--patch-size', str(int(patch_cfg['patch_size_px'])),
        '--upsample-factor', str(float(patch_cfg['upsample_factor'])),
        '--um-per-px', str(float(patch_cfg['he_um_per_px_override'])),
        '--max-cells-per-sample', str(quick_max_cells_per_sample),
        '--seed', '42',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT, check=True)
else:
    print('Reusing Cellpose prepared metadata:', prepare_summary)

meta_check = pd.read_parquet(source_dir / 'prepared_meta.parquet', columns=['sample_id', 'cell_id', 'transcript_counts', 'n_boundary_vertices'])
if meta_check['cell_id'].astype(str).str.startswith('CR').any():
    raise ValueError('Prepared source unexpectedly contains CR-prefixed IDs; expected Cellpose Xenium cell IDs')
print('Prepared full Cellpose-boundary cells:', meta_check.shape[0])
print(meta_check.groupby('sample_id')['transcript_counts'].agg(['size', 'min', 'median', lambda x: int((x < 20).sum())]).rename(columns={'<lambda_0>':'n_lt20'}).to_string())

if not he_summary.exists():
    cmd = [python, str(PROJECT / 'scripts/image_embedding/henium_custom_pipeline.py'), '--config', str(run_cfg), '--outdir', str(source_dir), '--step', 'embed-he', '--device', 'cuda', '--seed', '42']
    try:
        subprocess.run(cmd, cwd=PROJECT, check=True)
    except subprocess.CalledProcessError as e:
        raise RuntimeError('Fresh H-Optimus inference failed. Check CUDA/model cache or set HUGGINGFACE_HUB_TOKEN after getting access to bioptimus/H-optimus-1.') from e
else:
    print('Reusing Cellpose H-Optimus embeddings:', he_summary)
print(json.dumps(json.loads(prepare_summary.read_text()), indent=2))
print(json.dumps(json.loads(he_summary.read_text()), indent=2))


## 2. Full Cellpose Xenium transcript-count distribution

This QC plot checks the full Cellpose Xenium all-cell source. The count is `transcript_counts` from `cells.csv.gz`, and low-transcript cells are retained.


In [ ]:
cellpose_hist_csv = tables_dir / 'cellpose_transcript_count_distribution.csv'
cellpose_hist_png = figures_dir / 'cellpose_transcript_count_histogram.png'
meta_for_hist = pd.read_parquet(source_dir / 'prepared_meta.parquet', columns=['sample_id', 'cell_id', 'transcript_counts'])
summary_rows = []
for sid, g in meta_for_hist.groupby('sample_id'):
    tx = g['transcript_counts'].astype(float)
    summary_rows.append({
        'sample_id': sid,
        'n_cells': int(g.shape[0]),
        'min_transcripts': int(tx.min()),
        'q01': float(tx.quantile(0.01)),
        'q05': float(tx.quantile(0.05)),
        'q25': float(tx.quantile(0.25)),
        'median': float(tx.median()),
        'mean': float(tx.mean()),
        'max_transcripts': int(tx.max()),
        'n_lt20': int((tx < 20).sum()),
        'frac_lt20': float((tx < 20).mean()),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(cellpose_hist_csv, index=False)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, len(summary_rows), figsize=(6 * len(summary_rows), 4), squeeze=False)
for ax, (sid, g) in zip(axes.flat, meta_for_hist.groupby('sample_id')):
    vals = g['transcript_counts'].to_numpy()
    ax.hist(vals, bins=80, color='#4f6f8f', alpha=0.85)
    ax.axvline(20, color='crimson', linestyle='--', linewidth=1.2)
    ax.set_title(f'{sid}: transcript_counts')
    ax.set_xlabel('transcripts per cell')
    ax.set_ylabel('number of cells')
    ax.set_yscale('log')
fig.tight_layout()
fig.savefig(cellpose_hist_png, dpi=170)
plt.close(fig)
display(summary_df)
display(Image(filename=str(cellpose_hist_png)))


## 3. Prepare RNA-aligned BLEEP image embedding

`he_embeddings.npy` is the raw H-Optimus image embedding for the full Cellpose all-cell source. `aligned_image.npy` is created by aligning that raw image embedding to Xenium-normalized RNA PCA10 with the BLEEP-style soft-target projection model.

This notebook clusters only `aligned_image.npy`. RNA is used to train the image projection, but RNA is not directly used in the image-only Leiden graph in this notebook. The RNA + image WNN analysis is handled separately in Notebook 3.

In [ ]:
rna_pca10 = filtered_source / 'rna_xenium_norm_log1p_scale_pca10.npy'
rna_pca10_summary = filtered_source / 'rna_xenium_norm_log1p_scale_pca10_summary.json'
raw_he_embedding = filtered_source / 'he_embeddings.npy'
aligned_image = filtered_source / 'aligned_image.npy'
aligned_rna = filtered_source / 'aligned_rna.npy'
aligned_fused = filtered_source / 'aligned_fused.npy'
alignment_summary = filtered_source / 'alignment_summary.json'

if not (rna_pca10.exists() and rna_pca10_summary.exists()):
    cmd = [
        python, str(PROJECT / 'scripts/rna_wnn/compute_xenium_norm_log1p_scale_pca_allgenes.py'),
        '--source-dir', str(filtered_source),
        '--output-npy', rna_pca10.name,
        '--summary-json', rna_pca10_summary.name,
        '--n-components', str(int(hoptimus_cfg['rna_embedding']['pca_components'])),
        '--batch-size', '4096',
        '--clip', '10',
        '--seed', str(cfg['parameters']['random_seed']),
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT, check=True)
else:
    print('Reusing Xenium-normalized RNA PCA10:', rna_pca10)

reuse_alignment = False
if aligned_image.exists() and aligned_rna.exists() and aligned_fused.exists() and alignment_summary.exists():
    try:
        summary = json.loads(alignment_summary.read_text())
        reuse_alignment = (
            summary.get('method') == 'bleep_style_alignment_raw_hoptimus_to_xenium_norm_rna_pca'
            and Path(summary.get('image_embedding_path', '')).resolve() == raw_he_embedding.resolve()
            and Path(summary.get('rna_embedding_path', '')).resolve() == rna_pca10.resolve()
        )
    except Exception:
        reuse_alignment = False

if not reuse_alignment:
    cmd = [
        python, str(PROJECT / 'scripts/rna_wnn/align_raw_he_to_xenium_rna_bleep.py'),
        '--source-dir', str(filtered_source),
        '--image-embedding', str(raw_he_embedding),
        '--rna-embedding', str(rna_pca10),
        '--out-image', aligned_image.name,
        '--out-rna', aligned_rna.name,
        '--out-fused', aligned_fused.name,
        '--summary-json', alignment_summary.name,
        '--latent-dim', str(int(alignment_cfg['latent_dim'])),
        '--batch-size', str(int(alignment_cfg['batch_size'])),
        '--max-epochs', str(int(alignment_cfg['max_epochs'])),
        '--early-stop-patience', str(int(alignment_cfg['early_stop_patience'])),
        '--device', 'cuda',
        '--seed', '42',
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT, check=True)
else:
    print('Reusing direct Xenium-RNA-aligned BLEEP image embedding:', aligned_image)

meta_rows = pd.read_parquet(filtered_source / 'prepared_meta.parquet', columns=['sample_id', 'cell_id']).shape[0]
shapes = {
    'prepared_meta_rows': int(meta_rows),
    'he_embeddings': list(np.load(raw_he_embedding, mmap_mode='r').shape),
    'rna_pca10': list(np.load(rna_pca10, mmap_mode='r').shape),
    'aligned_image': list(np.load(aligned_image, mmap_mode='r').shape),
}
print(json.dumps(shapes, indent=2))

## 4. Image-only clustering: RNA-aligned BLEEP image embedding with Harmony

The clustered representation is the RNA-aligned BLEEP image embedding `aligned_image.npy`. The script row-wise L2-normalizes the embedding, applies Harmony by `sample_id`, L2-normalizes again after Harmony, builds a cosine kNN graph with `n_neighbors=30`, computes UMAP with `min_dist=0.3`, and scans Leiden resolution to target 40 image clusters.

This is still an image-only morphology clustering step. RNA is used only upstream to learn the BLEEP projection; RNA values are not part of the Leiden graph in this notebook.

In [ ]:
branch_specs = [
    {
        'name': 'rna_aligned_bleep_harmony',
        'label': 'RNA-aligned BLEEP image, Harmony by sample',
        'outdir': rna_aligned_cluster_dir,
        'embedding_file': 'aligned_image.npy',
        'pca_components': '0',
        'prefix': 'rna_aligned_bleep_harmony',
        'run_harmony': True,
    },
]
resolutions = '0.5,0.8,1.0,1.2,1.5,1.8,2.0,2.2,2.5,2.8,3.0,3.5,4.0'
branch_summaries = []
(image_results / 'qc').mkdir(parents=True, exist_ok=True)
for spec in branch_specs:
    outdir = spec['outdir']
    outdir.mkdir(parents=True, exist_ok=True)
    cluster_csv = outdir / 'joint_umap_clusters_target40.csv'
    summary_json = outdir / 'cluster_target40_summary.json'
    if cluster_csv.exists() and summary_json.exists():
        print('Reusing image branch:', spec['name'], cluster_csv)
    else:
        cmd = [
            python, str(PROJECT / 'scripts/image_embedding/run_i_graph_post_harmony_umap_only.py'),
            '--source-dir', str(filtered_source),
            '--outdir', str(outdir),
            '--embedding-file', spec['embedding_file'],
            '--pca-components', spec['pca_components'],
            '--n-neighbors', '30',
            '--metric', 'cosine',
            '--min-dist', '0.3',
            '--seed', '42',
            '--harmony-key', 'sample_id',
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, cwd=PROJECT, check=True, env={**os.environ, 'MPLBACKEND':'Agg'})
        cmd = [
            python, str(PROJECT / 'scripts/image_embedding/cluster_existing_umap_fast.py'),
            '--h5ad', str(outdir / 'joint_umap.h5ad'),
            '--target-clusters', '40',
            '--resolutions', resolutions,
            '--seed', '42',
        ]
        print(' '.join(cmd))
        subprocess.run(cmd, cwd=PROJECT, check=True, env={**os.environ, 'MPLBACKEND':'Agg'})
    summary = json.loads((outdir / 'cluster_target40_summary.json').read_text())
    summary['branch'] = spec['name']
    summary['label'] = spec['label']
    summary['embedding_file'] = spec['embedding_file']
    summary['pca_components'] = spec['pca_components']
    summary['run_harmony'] = True
    branch_summaries.append(summary)

    for name in ['joint_umap_clusters_target40.csv','embedding_cluster_sizes_target40.csv','embedding_clusters_target40.csv','joint_umap_leiden_target40_resolution_scan.csv','cluster_target40_summary.json']:
        shutil.copy2(outdir / name, tables_dir / f"{spec['prefix']}_{name}")
        shutil.copy2(outdir / name, tables_dir / name)
    for name in ['joint_umap_leiden_target40.png','joint_umap_qc_panels.png']:
        shutil.copy2(outdir / name, figures_dir / f"{spec['prefix']}_{name}")
        shutil.copy2(outdir / name, figures_dir / name)
    for name in ['i_graph_post_harmony.summary.json','umap_summary.json']:
        if (outdir / name).exists():
            shutil.copy2(outdir / name, image_results / 'qc' / name)

print(pd.DataFrame(branch_summaries).to_string(index=False))

## 5. Quick UMAP and QC visualization

In [ ]:
for spec in branch_specs:
    print(spec['label'])
    display(Image(filename=str(figures_dir / f"{spec['prefix']}_joint_umap_qc_panels.png")))
    display(Image(filename=str(figures_dir / f"{spec['prefix']}_joint_umap_leiden_target40.png")))


## 6. Export H&E patch review panels

The 50-cell per-cluster PDFs are for inline morphology review. The 200-cell PDFs are for external review. Both use the full all-cell clustering tables and draw dotted Cellpose `cell_boundaries.csv.gz` outlines when the selected polygon falls inside the H&E crop.


In [ ]:
def export_patch_pdf(cluster_csv, out_pdf, out_csv, n_patches, n_cols, draw_boundaries=True):
    cmd = [
        python, str(PROJECT / 'scripts/plotting/export_cluster_he_patch_pdf.py'),
        '--meta', str(source_dir / 'prepared_meta.parquet'),
        '--clusters', str(cluster_csv),
        '--out-pdf', str(out_pdf),
        '--out-csv', str(out_csv),
        '--patch-size', '224',
        '--upsample-factor', '4',
        '--n-patches-per-cluster', str(n_patches),
        '--n-cols', str(n_cols),
        '--seed', '42',
    ]
    if draw_boundaries:
        cmd += ['--boundaries-dir', str(boundaries_dir), '--draw-boundaries']
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT, check=True, env={**os.environ, 'MPLBACKEND':'Agg'})

spec = branch_specs[0]
prefix = spec['prefix']
cluster_csv_path = spec['outdir'] / 'joint_umap_clusters_target40.csv'
quick_pdf = patch_dir / f'{prefix}_target40_he_patches_50cells_per_cluster_boundaries.pdf'
quick_csv = patch_dir / f'{prefix}_target40_he_patches_50cells_per_cluster_boundaries.csv'
large_pdf = patch_dir / f'{prefix}_target40_he_patches_200cells_per_cluster_boundaries.pdf'
large_csv = patch_dir / f'{prefix}_target40_he_patches_200cells_per_cluster_boundaries.csv'
export_patch_pdf(cluster_csv_path, quick_pdf, quick_csv, 50, 10, True)
export_patch_pdf(cluster_csv_path, large_pdf, large_csv, 200, 10, True)

clusters = pd.read_csv(cluster_csv_path)
cluster34_csv = tables_dir / 'rna_aligned_bleep_harmony_target40_cluster34_cells.csv'
cluster34_cells = clusters[clusters['cluster'].astype(str) == '34'].copy()
cluster34_cells.to_csv(cluster34_csv, index=False)
cluster34_pdf = patch_dir / 'rna_aligned_bleep_harmony_target40_cluster34_he_patches_50cells_boundaries.pdf'
cluster34_patch_csv = patch_dir / 'rna_aligned_bleep_harmony_target40_cluster34_he_patches_50cells_boundaries.csv'
export_patch_pdf(cluster34_csv, cluster34_pdf, cluster34_patch_csv, 50, 10, True)

branch_patch_outputs = {
    prefix: {'quick_pdf': quick_pdf, 'quick_csv': quick_csv, 'large_pdf': large_pdf, 'large_csv': large_csv},
    'cluster34': {'quick_pdf': cluster34_pdf, 'quick_csv': cluster34_patch_csv, 'cluster_csv': cluster34_csv},
}
print(prefix, 'quick', pd.read_csv(quick_csv).shape, quick_pdf)
print(prefix, 'large', pd.read_csv(large_csv).shape, large_pdf)
print('cluster34', pd.read_csv(cluster34_patch_csv).shape, cluster34_pdf)

### Inline H&E patch review

The first PDF contains 50 randomly selected H&E patches per image cluster for the RNA-aligned BLEEP Harmony target40 result. The second PDF isolates cluster 34, the morphology-confirmed neutrophil cluster carried forward into Notebook 3.

In [ ]:
print('RNA-aligned BLEEP Harmony target40: 50 cells per cluster')
display(IFrame(src=str(quick_pdf), width='100%', height=700))
print('Cluster 34 only: 50 H&E patches with Cellpose cell-boundary overlays')
display(IFrame(src=str(cluster34_pdf), width='100%', height=700))

## 7. Per-sample cluster exports for Xenium-style review

In [ ]:
per_sample_root = image_results / 'per_sample_cluster_exports'
per_sample_outputs = []
for spec in branch_specs:
    outdir = per_sample_root / f"{spec['prefix']}_target40_allcells"
    cmd = [
        python, str(PROJECT / 'scripts/plotting/export_per_sample_cluster_groups.py'),
        '--clusters', str(spec['outdir'] / 'joint_umap_clusters_target40.csv'),
        '--outdir', str(outdir),
        '--run-name', f"{spec['prefix']}_target40_allcells",
    ]
    print(' '.join(cmd))
    subprocess.run(cmd, cwd=PROJECT, check=True)
    per_sample_outputs.append({'branch': spec['prefix'], 'outdir': outdir})
print(pd.DataFrame(per_sample_outputs).to_string(index=False))

## 8. CXCL8 expression plots and CXCL8/CXCR2 dotplots for the RNA-aligned BLEEP Harmony image clusters

These plots use Xenium size-factor normalization to global median library size followed by `log1p`. The CXCL8 UMAP highlights whether the morphology-derived clusters correspond to neutrophil-like chemokine signal. The cluster dotplot summarizes both mean normalized expression and the percentage of cells with a raw count greater than zero.

In [ ]:
spec = branch_specs[0]
prefix = spec['prefix']
out_csv = tables_dir / f'{prefix}_target40_CXCL8_umap_expression.csv'
out_png = figures_dir / f'{prefix}_target40_CXCL8_umap_expression.png'
out_summary = tables_dir / f'{prefix}_target40_CXCL8_umap_expression_summary.json'
cmd = [
    python, str(PROJECT / 'scripts/plotting/plot_gene_expression_umap.py'),
    '--source-dir', str(filtered_source),
    '--clusters', str(spec['outdir'] / 'joint_umap_clusters_target40.csv'),
    '--branch-name', spec['name'],
    '--gene', 'CXCL8',
    '--out-csv', str(out_csv),
    '--out-png', str(out_png),
    '--out-summary', str(out_summary),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT, check=True, env={**os.environ, 'MPLBACKEND':'Agg'})
display(Image(filename=str(out_png)))

out_dot_csv = tables_dir / f'{prefix}_target40_CXCL8_CXCR2_dotplot_values.csv'
out_dot_png = figures_dir / f'{prefix}_target40_CXCL8_CXCR2_dotplot.png'
out_dot_summary = tables_dir / f'{prefix}_target40_CXCL8_CXCR2_dotplot_summary.json'
cmd = [
    python, str(PROJECT / 'scripts/plotting/plot_cluster_gene_dotplot.py'),
    '--source-dir', str(filtered_source),
    '--clusters', str(spec['outdir'] / 'joint_umap_clusters_target40.csv'),
    '--branch-name', spec['name'],
    '--genes', 'CXCL8,CXCR2',
    '--out-csv', str(out_dot_csv),
    '--out-png', str(out_dot_png),
    '--out-summary', str(out_dot_summary),
]
print(' '.join(cmd))
subprocess.run(cmd, cwd=PROJECT, check=True, env={**os.environ, 'MPLBACKEND':'Agg'})
display(Image(filename=str(out_dot_png)))

dot = pd.read_csv(out_dot_csv)
cxcl8 = dot[dot['gene'] == 'CXCL8'].copy()
cxcl8['mean_rank_desc'] = cxcl8['mean_log1p_global_median_norm'].rank(ascending=False, method='min').astype(int)
cxcl8['pct_rank_desc'] = cxcl8['pct_expressing_raw_count_gt0'].rank(ascending=False, method='min').astype(int)
cluster34_cxcl8 = cxcl8[cxcl8['cluster'].astype(str) == '34'].copy()
cluster34_summary_path = tables_dir / 'rna_aligned_bleep_harmony_cluster34_CXCL8_summary.csv'
cluster34_cxcl8.to_csv(cluster34_summary_path, index=False)
print('Cluster 34 CXCL8 summary:')
display(cluster34_cxcl8)
print('Top CXCL8 clusters by mean normalized expression:')
display(cxcl8.sort_values(['mean_log1p_global_median_norm', 'pct_expressing_raw_count_gt0'], ascending=False).head(10))

## 9. Cluster 34 dense H&E regions

The patch PDF shows individual cells. This section also searches each sample for local regions enriched for cluster 34 cells and saves two zoomed H&E regions per sample with cluster 34 centroids marked. This provides spatial context for areas where multiple morphology-confirmed neutrophils appear together.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from sklearn.neighbors import NearestNeighbors
import tifffile

meta_cols = ['sample_id', 'cell_id', 'x_px', 'y_px', 'he_image']
meta = pd.read_parquet(source_dir / 'prepared_meta.parquet', columns=meta_cols)
clusters = pd.read_csv(rna_aligned_cluster_dir / 'joint_umap_clusters_target40.csv')
cluster34 = clusters[clusters['cluster'].astype(str) == '34'][['sample_id', 'cell_id', 'cluster']].merge(meta, on=['sample_id', 'cell_id'], how='left')
region_records = []
crop_size = 1400
radius = 700
for sample_id, g in cluster34.dropna(subset=['x_px', 'y_px']).groupby('sample_id', sort=True):
    coords = g[['x_px', 'y_px']].to_numpy(dtype=float)
    if coords.shape[0] == 0:
        continue
    nn = NearestNeighbors(radius=radius)
    nn.fit(coords)
    neigh = nn.radius_neighbors(coords, return_distance=False)
    counts = np.array([len(x) for x in neigh])
    candidate_order = np.argsort(-counts)
    centers = []
    for idx in candidate_order:
        center = coords[idx]
        if all(np.linalg.norm(center - c) > crop_size for c in centers):
            centers.append(center)
        if len(centers) == 2:
            break
    he_path = Path(g['he_image'].iloc[0])
    img = tifffile.imread(he_path)
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.ndim == 3 and img.shape[0] in (3, 4) and img.shape[-1] not in (3, 4):
        img = np.moveaxis(img[:3], 0, -1)
    img = np.asarray(img[..., :3])
    if img.dtype != np.uint8:
        img = np.clip(img, 0, 255).astype(np.uint8)
    h, w = img.shape[:2]
    for region_idx, center in enumerate(centers, start=1):
        cx, cy = center
        x0 = int(max(0, min(w - crop_size, round(cx - crop_size / 2))))
        y0 = int(max(0, min(h - crop_size, round(cy - crop_size / 2))))
        x1 = min(w, x0 + crop_size)
        y1 = min(h, y0 + crop_size)
        crop = img[y0:y1, x0:x1]
        in_crop = g[(g['x_px'] >= x0) & (g['x_px'] < x1) & (g['y_px'] >= y0) & (g['y_px'] < y1)].copy()
        fig, ax = plt.subplots(figsize=(7.5, 7.5), dpi=170)
        ax.imshow(crop)
        for _, row in in_crop.iterrows():
            ax.add_patch(Circle((float(row['x_px']) - x0, float(row['y_px']) - y0), radius=24, fill=False, edgecolor='cyan', linewidth=1.4, linestyle=(0, (2, 2))))
        ax.set_title(f'{sample_id} cluster 34 dense region {region_idx}: {len(in_crop)} cells', fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        out_png = figures_dir / f'rna_aligned_bleep_harmony_cluster34_dense_region_{sample_id}_{region_idx}.png'
        fig.tight_layout()
        fig.savefig(out_png, bbox_inches='tight')
        plt.close(fig)
        region_records.append({'sample_id': sample_id, 'region': region_idx, 'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1, 'cluster34_cells_in_crop': int(len(in_crop)), 'figure': str(out_png)})
    del img

dense_regions = pd.DataFrame(region_records)
dense_regions_path = tables_dir / 'rna_aligned_bleep_harmony_cluster34_dense_regions.csv'
dense_regions.to_csv(dense_regions_path, index=False)
display(dense_regions)
for fig_path in dense_regions['figure']:
    display(Image(filename=fig_path))

## 10. Manual neutrophil review table

In [ ]:
cluster_sizes = tables_dir / 'embedding_cluster_sizes_target40.csv'
review_table = tables_dir / 'neutrophil_cluster_review.csv'
cmd = [python, str(PROJECT / 'scripts/image_embedding/make_neutrophil_review_template.py'), '--cluster-sizes', str(cluster_sizes), '--out', str(review_table)]
subprocess.run(cmd, cwd=PROJECT, check=True)
review = pd.read_csv(review_table)
review['confirmed_neutrophil'] = review['image_cluster'].astype(str).eq('34')
review['review_note'] = np.where(review['confirmed_neutrophil'], 'confirmed neutrophil from H&E morphology and CXCL8 signal', '')
review.to_csv(review_table, index=False)
display(review[review['confirmed_neutrophil']].head())
display(review.head(10))

## 11. Export confirmed neutrophil cells after review

In [ ]:
confirmed_out = tables_dir / 'confirmed_neutrophil_cells.csv'
cluster_cells = rna_aligned_cluster_dir / 'joint_umap_clusters_target40.csv'
cells = pd.read_csv(cluster_cells)
review = pd.read_csv(review_table)
review['confirmed_neutrophil'] = review['confirmed_neutrophil'].astype(str).str.lower().str.strip()
keep_clusters = set(review.loc[review['confirmed_neutrophil'].isin(['yes','y','true','1']), 'image_cluster'].astype(str))
confirmed = cells[cells['cluster'].astype(str).isin(keep_clusters)].copy()
confirmed['label_source'] = 'cluster 34 RNA-aligned BLEEP Harmony H&E morphology review with CXCL8 support'
confirmed['confirmed_neutrophil'] = True
confirmed.to_csv(confirmed_out, index=False)
print(f'wrote {confirmed_out} with {len(confirmed):,} cells from clusters {sorted(keep_clusters)}')
display(confirmed.head())

## 12. Output manifest

In [ ]:
run_summary = {
    'registration': 'Palom',
    'hoptimus_inference': 'reused if he_embeddings.npy already existed; otherwise generated from Palom registered H&E patches centered on Cellpose cell-boundary centroids',
    'cell_source': 'Cellpose Xenium cell_boundaries.csv.gz polygons joined to cell_feature_matrix RNA counts',
    'image_reduction': 'RNA-aligned BLEEP aligned_image.npy only. The embedding is L2-normalized, Harmony-corrected by sample_id, L2-normalized again, then clustered with a cosine kNN graph, UMAP, and Leiden target40.',
    'run_harmony': True,
    'confirmed_neutrophil_default': 'cluster 34',
    'min_transcript_filtering': 'not applied in Notebook 2; Notebook 3 filters non-neutrophils after rescuing cluster34 neutrophils',
    'outputs': {
        'rna_aligned_bleep_harmony_dir': str(rna_aligned_cluster_dir),
        'dotplot_table': str(tables_dir / 'rna_aligned_bleep_harmony_target40_CXCL8_CXCR2_dotplot_values.csv'),
        'cxcl8_umap_expression_table': str(tables_dir / 'rna_aligned_bleep_harmony_target40_CXCL8_umap_expression.csv'),
        'cluster34_patch_pdf': str(cluster34_pdf),
        'cluster34_dense_regions': str(tables_dir / 'rna_aligned_bleep_harmony_cluster34_dense_regions.csv'),
        'per_sample_cluster_exports': str(image_results / 'per_sample_cluster_exports' / 'rna_aligned_bleep_harmony_target40_allcells'),
    },
    'rna_image_wnn': 'not run in Notebook 2; Notebook 3 performs Muon WNN after cluster34 neutrophil rescue',
}
(image_results / 'run_summary.json').write_text(json.dumps(run_summary, indent=2))
files = sorted([p for p in image_results.rglob('*') if p.is_file()])
manifest = {'result_dir': str(image_results), 'n_files': len(files), 'files': [{'path': str(p), 'size_bytes': p.stat().st_size} for p in files]}
manifest_path = image_results / 'output_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
print(json.dumps({'manifest': str(manifest_path), 'n_files': len(files)}, indent=2))

## 13. Output guide

Notebook 2 writes full Cellpose all-cell RNA-aligned image morphology outputs under `results/02_hoptimus_image_clusters/`.

- `embeddings/hoptimus_cellpose_ap2320a_ap0921a_source_allcells/`: prepared Cellpose metadata/counts plus H-Optimus embeddings from H&E patches centered on `cell_boundaries.csv.gz` centroids. It also contains Xenium-normalized RNA PCA10 and the BLEEP-aligned image embedding used here.
- `tables/cellpose_boundaries_allcells/`: polygon vertices converted from Cellpose `cell_boundaries.csv.gz` for dotted patch overlays.
- `tables/cellpose_transcript_count_distribution.csv` and `figures/cellpose_transcript_count_histogram.png`: confirms low-transcript cells are present and retained in the all-cell source.
- `image_rna_aligned_bleep_harmony_target40_allcells/`: the only Notebook 2 clustering branch. It contains the image-only UMAP and Leiden target40 outputs from `aligned_image.npy` with Harmony by `sample_id`.
- `per_sample_cluster_exports/rna_aligned_bleep_harmony_target40_allcells/`: per-sample `cell_id,group` exports generated from the same cluster CSV as the patch PDFs.
- `he_patches/rna_aligned_bleep_harmony_target40_he_patches_50cells_per_cluster_boundaries.pdf`: inline morphology reassessment PDF with 50 cells per cluster and dotted Cellpose boundaries.
- `he_patches/rna_aligned_bleep_harmony_target40_he_patches_200cells_per_cluster_boundaries.pdf`: larger external review PDF with the same dotted Cellpose boundaries.
- `he_patches/rna_aligned_bleep_harmony_target40_cluster34_he_patches_50cells_boundaries.pdf`: cluster 34-only H&E patch review, displayed inline.
- `figures/rna_aligned_bleep_harmony_target40_CXCL8_umap_expression.png` and `tables/rna_aligned_bleep_harmony_cluster34_CXCL8_summary.csv`: show that cluster 34 has high CXCL8 signal relative to other image clusters.
- `figures/rna_aligned_bleep_harmony_cluster34_dense_region_*.png`: two zoomed H&E regions per sample with multiple cluster 34 cells marked.
- `tables/neutrophil_cluster_review.csv`: review table with cluster 34 marked as confirmed neutrophil for Notebook 3.
- `tables/confirmed_neutrophil_cells.csv`: all cells in confirmed cluster 34, carried into Notebook 3 before non-neutrophil transcript filtering.